# AI Trader — TFT Forecaster (Transformer-based 5-day Price Prediction)

**Purpose:** Train a Temporal Fusion Transformer–inspired model that predicts
the next 5 trading days' prices for any NSE symbol.

Architecture: Multi-head self-attention Transformer encoder over a 60-day
OHLCV feature window → linear head outputting 5 daily log-returns.

**Colab setup:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells (Ctrl+F9)
3. Copy the **Google Drive File ID** printed in the last cell
4. Add it to your `.env`: `TFT_GDRIVE_ID=<paste_here>`

**Training time:** ~10–20 min on T4 GPU for 50 symbols × 500 days.

In [ ]:
!pip install -q yfinance torch numpy pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════

DRIVE_SAVE_DIR    = '/content/drive/MyDrive/ai_trader_models'
HISTORY_PERIOD    = '3y'

# Must stay in sync with backend/app/services/tft_service.py _build_model
SEQ_LEN           = 60
INPUT_SIZE        = 5     # log_ret, high_pct, low_pct, vol_ratio, rsi_norm
D_MODEL           = 64
NHEAD             = 4
NUM_LAYERS        = 2
FORECAST_HORIZON  = 5     # predict next 5 trading days

EPOCHS     = 80
BATCH_SIZE = 64
LR         = 1e-3

# Minimum bars required to include a symbol in training
MIN_BARS    = SEQ_LEN + FORECAST_HORIZON + 20

# Cap total symbols (None = all qualifying; reduce if you hit Colab timeout)
MAX_SYMBOLS = None


In [ ]:
# ── Fetch full NSE equity universe from NSE public CSV ───────────────────────
# NSE lists ~1800+ actively traded equities. We download all of them and filter
# to those with at least MIN_BARS of price history (set in CONFIG above).
# Typical qualifying count after filtering: 600–900 symbols.
# Colab T4 GPU training time: ~20–40 min (LSTM), ~35–60 min (TFT).
import io as _io, urllib.request as _req
import pandas as pd

NSE_EQUITY_CSV = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"

try:
    _r = _req.Request(NSE_EQUITY_CSV, headers={"User-Agent": "Mozilla/5.0"})
    with _req.urlopen(_r, timeout=30) as resp:
        content = resp.read().decode("utf-8", errors="ignore")
    df_universe = pd.read_csv(_io.StringIO(content))
    raw_syms = df_universe["SYMBOL"].dropna().str.strip().tolist()
    SYMBOLS  = [s + ".NS" for s in raw_syms if s]
    print(f"NSE universe: {len(SYMBOLS)} symbols fetched from NSE CSV")
except Exception as _e:
    print(f"WARNING: NSE CSV unavailable ({_e}). Using ~200-symbol fallback list.")
    _FALLBACK = [
        "RELIANCE","TCS","HDFCBANK","INFY","ICICIBANK","HINDUNILVR","ITC","SBIN",
        "BHARTIARTL","KOTAKBANK","LT","AXISBANK","ASIANPAINT","MARUTI","SUNPHARMA",
        "WIPRO","ULTRACEMCO","TITAN","BAJFINANCE","ONGC","NTPC","POWERGRID",
        "NESTLEIND","TECHM","HCLTECH","INDUSINDBK","COALINDIA","GRASIM","JSWSTEEL",
        "TATAMOTORS","ADANIENT","ADANIPORTS","BAJAJ-AUTO","BPCL","CIPLA","DIVISLAB",
        "DRREDDY","EICHERMOT","HEROMOTOCO","HINDALCO","M&M","BRITANNIA","APOLLOHOSP",
        "TATASTEEL","UPL","VEDL","BAJAJFINSV","HDFCLIFE","SBILIFE","PIDILITIND",
        "HAVELLS","MUTHOOTFIN","IDFCFIRSTB","BANKBARODA","PNB","CANBK","FEDERALBNK",
        "BANDHANBNK","RBLBANK","SIEMENS","ABB","VOLTAS","AMBUJACEM","SHREECEM",
        "DALMIACEM","JKCEMENT","ZOMATO","NYKAA","IRCTC","DELHIVERY","NAUKRI",
        "TRENT","ABFRL","PAGEIND","MANYAVAR","BATAINDIA","METRO","RELAXO",
        "GLAND","GRANULES","NATCO","LALPATHLAB","METROPOLIS","THYROCARE","ALKEM",
        "AUROPHARMA","TORNTPHARM","IPCALAB","PFIZER","GLAXO","ABBOTINDIA","BIOCON",
        "TATACHEM","DEEPAKNTR","NAVINFLUOR","NOCIL","SRF","ATUL","GHCL",
        "AAVAS","HOMEFIRST","APTUS","CANFINHOME","LICHSGFIN","PNBHOUSING",
        "MAHINDCIE","MOTHERSON","BOSCHLTD","BHARATFORG","SUNDARMFIN","CUMMINSIND",
        "THERMAX","ESCORTS","SONACOMS","MINDA","ENDURANCE","GABRIEL",
        "MPHASIS","LTIM","PERSISTENT","COFORGE","CYIENT","KPIT","OFSS",
        "INTERGLOBE","CONCOR","BLUEDART","TATACOMM","ZEEL","SUNTV","STAR",
        "LICI","PEL","CHOLAFIN","MANAPPURAM","M&MFIN","SRTRANSFIN",
        "SHRIRAMFIN","ICICIGI","ICICIPRU","HDFCAMC","NIPPONLIFE",
        "ANGELONE","MOTILALOFS","EDELWEISS","LAURUSLABS","DIVIS","GLENMARK",
        "APOLLOHOSP","FORTIS","MAXHEALTH","NARAYANA","RAINBOW",
    ]
    SYMBOLS = [s + ".NS" for s in _FALLBACK]

if MAX_SYMBOLS:
    SYMBOLS = SYMBOLS[:MAX_SYMBOLS]

print(f"Will attempt to download: {len(SYMBOLS)} symbols")
print("Symbols with < MIN_BARS of data will be skipped automatically.")


In [ ]:
import os, math
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import yfinance as yf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Fetch data ───────────────────────────────────────────────────────────────
print(f'Fetching {len(SYMBOLS)} symbols...')
all_data = {}
failed   = []

for sym in tqdm(SYMBOLS):
    try:
        df = yf.download(sym, period=HISTORY_PERIOD,
                         interval='1d', auto_adjust=True, progress=False)
        df.dropna(inplace=True)
        if len(df) >= SEQ_LEN + FORECAST_HORIZON + 20:
            all_data[sym] = df
        else:
            failed.append(sym)
    except Exception as e:
        failed.append(sym)

print(f'Loaded {len(all_data)} symbols  |  Failed: {failed}')

In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
# Features (must match backend/app/services/tft_service.py _build_sequence):
#   0  log_ret    — log daily return
#   1  high_pct   — (high − close) / close
#   2  low_pct    — (low  − close) / close
#   3  vol_ratio  — volume / rolling-20-day-mean-volume
#   4  rsi_norm   — RSI(14) / 100

def rsi_series(log_rets, period=14):
    n   = len(log_rets)
    out = np.full(n, 50.0, dtype=np.float32)
    if n < period + 1:
        return out
    gains  = np.where(log_rets > 0, log_rets, 0.0)
    losses = np.where(log_rets < 0, -log_rets, 0.0)
    ag = gains[:period].mean()
    al = losses[:period].mean()
    for i in range(period, n):
        ag      = (ag * (period - 1) + gains[i])  / period
        al      = (al * (period - 1) + losses[i]) / period
        out[i]  = 100.0 - 100.0 / (1.0 + ag / (al + 1e-8))
    return out


def build_features(df: pd.DataFrame):
    closes  = df['Close'].values.squeeze().astype(np.float32)
    highs   = df['High'].values.squeeze().astype(np.float32)
    lows    = df['Low'].values.squeeze().astype(np.float32)
    volumes = df['Volume'].values.squeeze().astype(np.float32) + 1e-8

    log_ret  = np.diff(np.log(closes + 1e-8))
    high_pct = (highs[1:] - closes[1:]) / (closes[1:] + 1e-8)
    low_pct  = (lows[1:]  - closes[1:]) / (closes[1:] + 1e-8)

    vol_ma   = pd.Series(volumes).rolling(20, min_periods=1).mean().values[1:]
    vol_ratio = volumes[1:] / (vol_ma + 1e-8)

    rsi_norm = rsi_series(log_ret) / 100.0

    features = np.stack([log_ret, high_pct, low_pct, vol_ratio, rsi_norm], axis=1)
    log_rets_raw = log_ret   # used for labels
    return features.astype(np.float32), log_rets_raw.astype(np.float32)


def make_xy(features, log_rets, seq_len, horizon):
    X, y = [], []
    for i in range(len(features) - seq_len - horizon + 1):
        X.append(features[i : i + seq_len])
        y.append(log_rets[i + seq_len : i + seq_len + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


all_X, all_y = [], []
for sym, df in all_data.items():
    feat, lr = build_features(df)
    X, y     = make_xy(feat, lr, SEQ_LEN, FORECAST_HORIZON)
    all_X.append(X)
    all_y.append(y)

all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)
print(f'X shape: {all_X.shape}  y shape: {all_y.shape}')

In [ ]:
# ── Standardise features ─────────────────────────────────────────────────────
flat        = all_X.reshape(-1, INPUT_SIZE)
scaler_mean = flat.mean(axis=0)
scaler_std  = flat.std(axis=0) + 1e-8
all_X_norm  = (all_X - scaler_mean) / scaler_std

# Train / val split  (80 / 20)
n_train  = int(len(all_X_norm) * 0.8)
X_train  = torch.tensor(all_X_norm[:n_train])
y_train  = torch.tensor(all_y[:n_train])
X_val    = torch.tensor(all_X_norm[n_train:])
y_val    = torch.tensor(all_y[n_train:])

train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),
                          batch_size=BATCH_SIZE)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

In [ ]:
# ── Model definition ─────────────────────────────────────────────────────────
# MUST be identical to backend/app/services/tft_service.py _build_model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10_000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TFTForecaster(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, forecast_horizon):
        super().__init__()
        self.input_proj  = nn.Linear(input_size, d_model)
        self.pos_enc     = PositionalEncoding(d_model)
        enc_layer        = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4, dropout=0.1, batch_first=True,
        )
        self.transformer    = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.forecast_head  = nn.Linear(d_model, forecast_horizon)

    def forward(self, x):
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.transformer(h)
        return self.forecast_head(h[:, -1, :])


model = TFTForecaster(INPUT_SIZE, D_MODEL, NHEAD, NUM_LAYERS, FORECAST_HORIZON).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
                 optimizer, patience=6, factor=0.5)
criterion  = nn.MSELoss()

train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in tqdm(range(1, EPOCHS + 1)):
    model.train()
    t_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item() * len(xb)
    t_loss /= len(X_train)

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            v_loss += criterion(model(xb), yb).item() * len(xb)
    v_loss /= len(X_val)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    scheduler.step(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), '/tmp/tft_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}  train={t_loss:.7f}  val={v_loss:.7f}')

model.load_state_dict(torch.load('/tmp/tft_best.pt', map_location=DEVICE))
print(f'\nBest val loss: {best_val_loss:.7f}')

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train MSE')
plt.plot(val_losses,   label='Val MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend()
plt.title('TFT Forecaster — Training Loss')
plt.tight_layout(); plt.show()

# Directional accuracy on val set
model.eval()
pred_list, true_list = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        pred_list.append(model(xb.to(DEVICE)).cpu().numpy())
        true_list.append(yb.numpy())

preds = np.concatenate(pred_list)
trues = np.concatenate(true_list)

# Directional accuracy: did we predict the right sign for next-day return?
dir_acc_d1 = float((np.sign(preds[:, 0]) == np.sign(trues[:, 0])).mean())
print(f'Day-1 directional accuracy: {dir_acc_d1:.3f}')

# Sample forecast vs actual (first val symbol)
idx = 0
plt.figure(figsize=(8, 3))
plt.plot(trues[idx],  marker='o', label='Actual log-return')
plt.plot(preds[idx],  marker='s', label='Predicted log-return')
plt.xlabel('Day ahead'); plt.legend()
plt.title('Sample 5-day forecast')
plt.tight_layout(); plt.show()

In [ ]:
# ── Save artifact to Google Drive ─────────────────────────────────────────────
VERSION   = f"tft-v{datetime.now().strftime('%Y%m%d-%H%M')}"
SAVE_PATH = f'{DRIVE_SAVE_DIR}/tft_forecaster_latest.pt'

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

model.to('cpu')
payload = {
    'version': VERSION,
    'config': {
        'input_size':       INPUT_SIZE,
        'd_model':          D_MODEL,
        'nhead':            NHEAD,
        'num_layers':       NUM_LAYERS,
        'seq_len':          SEQ_LEN,
        'forecast_horizon': FORECAST_HORIZON,
    },
    'state_dict':    model.state_dict(),
    'scaler_mean':   scaler_mean.tolist(),
    'scaler_std':    scaler_std.tolist(),
    'feature_names': ['log_ret', 'high_pct', 'low_pct', 'vol_ratio', 'rsi_norm'],
    'metrics': {
        'best_val_mse':      round(float(best_val_loss), 8),
        'dir_acc_day1':      round(dir_acc_d1, 4),
        'n_symbols':         len(all_data),
        'forecast_horizon':  FORECAST_HORIZON,
    },
}
torch.save(payload, SAVE_PATH)
print(f'Saved to: {SAVE_PATH}')
print(f'Size: {os.path.getsize(SAVE_PATH) / 1024:.1f} KB')

In [ ]:
# ── Get Google Drive file ID and print instructions ───────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
creds   = GoogleCredentials.get_application_default()
service = build('drive', 'v3', credentials=creds)

results = service.files().list(
    q="name='tft_forecaster_latest.pt'",
    fields='files(id, name, webViewLink)',
).execute()

files = results.get('files', [])
if files:
    file_id   = files[0]['id']
    view_link = files[0]['webViewLink']

    service.permissions().create(
        fileId=file_id,
        body={'type': 'anyone', 'role': 'reader'},
    ).execute()

    print('=' * 60)
    print('TFT Forecaster artifact saved and shared.')
    print(f'\nGoogle Drive File ID: {file_id}')
    print(f'View link:           {view_link}')
    print(f'Day-1 directional accuracy: {dir_acc_d1:.1%}')
    print('\n>>> Add to your .env file:')
    print(f'TFT_GDRIVE_ID={file_id}')
    print('\nThen run inside the backend container:')
    print('  docker compose exec backend python scripts/download_models.py --tft-only')
    print('  docker compose restart backend celery-worker')
    print('=' * 60)
else:
    print('File not found in Drive — check DRIVE_SAVE_DIR path above.')